In [ ]:
# Route visualization notebook for GPS ping exports from this project
# If needed, install dependencies once:
# %pip install pandas folium matplotlib

from pathlib import Path
import json
import math

import pandas as pd
import folium
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 1) Load the latest GPS export JSON
This cell finds the newest `gps-pings-*.json` file in the project folder and loads it into a DataFrame.

In [ ]:
data_dir = Path(".")
json_files = sorted(
    data_dir.glob("gps-pings-*.json"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not json_files:
    raise FileNotFoundError(
        "No gps-pings-*.json file found in the current folder. "
        "Download one from the website first, then rerun this cell."
    )

data_file = json_files[0]
print(f"Using file: {data_file}")

with data_file.open("r", encoding="utf-8") as f:
    payload = json.load(f)

pings = payload.get("pings", [])
if not pings:
    raise ValueError("The JSON file has no pings to visualize.")

df = pd.DataFrame(pings)
print(f"Loaded {len(df)} pings")
display(df.head())

## 2) Clean, sort, and compute route distance
This cell parses timestamps, removes invalid rows, and calculates segment + cumulative distance in meters.

In [ ]:
required = ["latitude", "longitude", "timestampIso"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns in export: {missing}")

route_df = df.copy()
route_df["timestamp"] = pd.to_datetime(route_df["timestampIso"], errors="coerce", utc=True)
route_df = route_df.dropna(subset=["latitude", "longitude", "timestamp"])
route_df = route_df.sort_values("timestamp").reset_index(drop=True)

if len(route_df) < 2:
    raise ValueError("Need at least 2 valid points to draw a route.")


def haversine_m(lat1, lon1, lat2, lon2):
    r = 6_371_000.0  # Earth radius in meters
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    return 2 * r * math.atan2(math.sqrt(a), math.sqrt(1 - a))

segment_m = [0.0]
for i in range(1, len(route_df)):
    prev = route_df.iloc[i - 1]
    curr = route_df.iloc[i]
    d = haversine_m(prev.latitude, prev.longitude, curr.latitude, curr.longitude)
    segment_m.append(d)

route_df["segment_m"] = segment_m
route_df["cumulative_km"] = route_df["segment_m"].cumsum() / 1000

trip_duration = route_df["timestamp"].iloc[-1] - route_df["timestamp"].iloc[0]
total_distance_km = route_df["cumulative_km"].iloc[-1]

print(f"Points used: {len(route_df)}")
print(f"Total route distance: {total_distance_km:.3f} km")
print(f"Trip duration: {trip_duration}")
display(route_df[["timestamp", "latitude", "longitude", "segment_m", "cumulative_km"]].head())

## 3) Render the route on an interactive map
The polyline connects each geopoint in time order, with start/end markers and optional point markers.

In [ ]:
coords = route_df[["latitude", "longitude"]].values.tolist()
center = [route_df["latitude"].mean(), route_df["longitude"].mean()]

route_map = folium.Map(location=center, zoom_start=14, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=coords,
    color="#00e0b8",
    weight=5,
    opacity=0.9,
    tooltip="Driven route",
).add_to(route_map)

start = coords[0]
end = coords[-1]

folium.Marker(
    location=start,
    popup="Start",
    tooltip="Start",
    icon=folium.Icon(color="green", icon="play"),
).add_to(route_map)

folium.Marker(
    location=end,
    popup="End",
    tooltip="End",
    icon=folium.Icon(color="red", icon="stop"),
).add_to(route_map)

# Add a marker every N points to keep large routes readable.
step = max(1, len(route_df) // 30)
for i in range(0, len(route_df), step):
    row = route_df.iloc[i]
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=3,
        color="#8ce6ff",
        fill=True,
        fill_opacity=0.7,
        popup=f"#{i+1} | {row.timestamp}",
    ).add_to(route_map)

display(route_map)

## 4) Optional: save map to HTML
Use this if you want to share or open the route map outside the notebook.

In [ ]:
output_map = data_file.with_suffix(".route-map.html")
route_map.save(output_map)
print(f"Saved interactive map to: {output_map}")